# RAY-IMAGE v0.1 — First Working Prototype

This notebook uses a Colab GPU to train the prototype in two stages: VAE reconstruction, then a text-conditioned latent flow generator. It finishes by generating an image from a text prompt.

Select **Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU is attached. Select a GPU runtime and reconnect.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

In [ ]:
%cd /content
!rm -rf anime-ai-companion
!git clone https://github.com/Rishidev-20thcenturey/anime-ai-companion.git
%cd /content/anime-ai-companion
!pip install -q -r requirements.txt

In [ ]:
!python -m ray_image.train_smoke

In [ ]:
!python tools/make_toy_dataset.py --output data/toy --samples 256 --size 64

In [ ]:
!python -m ray_image.train_vae --manifest data/toy/manifest.jsonl --steps 1000 --batch-size 16 --save /content/ray_vae_v0_1.pt

In [ ]:
!python -m ray_image.train_generator --manifest data/toy/manifest.jsonl --vae /content/ray_vae_v0_1.pt --steps 2000 --batch-size 8 --save /content/ray_image_v0_1_trained.pt

In [ ]:
!python -m ray_image.generate --checkpoint /content/ray_image_v0_1_trained.pt --prompt 'a blue circle' --steps 40 --seed 42 --output /content/first_rays_image.png

In [ ]:
from IPython.display import display
from PIL import Image
from pathlib import Path
image_path = Path('/content/first_rays_image.png')
print('generated:', image_path.exists())
if image_path.exists():
    display(Image.open(image_path))

In [ ]:
from pathlib import Path
for path in ['/content/ray_vae_v0_1.pt', '/content/ray_image_v0_1_trained.pt', '/content/first_rays_image.png']:
    p = Path(path)
    print(p.name, 'exists=', p.exists(), 'size=', round(p.stat().st_size / 1024**2, 2) if p.exists() else '-')

## Prototype milestone

If the final image is produced, RAY-IMAGE has completed the full learned path: text → text encoder → DiT flow field → latent → VAE decoder → image. This toy run is an architecture proof, not a quality benchmark. The next stage is a real captioned anime dataset and longer training.